# CodeMoldova — Week 2 Wednesday — Answer key

REST Countries lab + Challenge. Try first, then check here.


In [ ]:
import requests
from pathlib import Path
import json


## 2 · JSON practice


In [ ]:
mini = {"name": {"common": "Moldova"}, "capital": ["Chișinău"], "population": 2_749_076, "borders": ["ROU", "UKR"], "languages": {"ron": "Romanian"}}
print(mini["languages"])


## 5 · Moldova — practice (example: Italy)


In [ ]:
FIELDS = "name,capital,population,languages,borders"
r = requests.get("https://restcountries.com/v3.1/name/italy?fields=FIELDS", timeout=10)
d = r.json()[0]
print(d["name"]["common"], d["capital"][0], d["population"])


## 6 · Neighbors from Moldova borders


In [ ]:
FIELDS = "name,population,capital"
r = requests.get("https://restcountries.com/v3.1/name/moldova?fields=borders", timeout=10)
for code in r.json()[0]["borders"]:
    n = requests.get(f"https://restcountries.com/v3.1/alpha/{code}?fields={FIELDS}", timeout=10).json()
    print(n["name"]["common"], n["population"], n["capital"][0])


## 7 · Three countries


In [ ]:
for c in ["moldova", "france", "japan"]:
    r = requests.get(f"https://restcountries.com/v3.1/name/{c}?fields=name,population", timeout=10)
    d = r.json()[0]
    print(d["name"]["common"], d["population"])


## 10 · Challenge


In [ ]:
FIELDS = "name,capital,population,languages,borders,flags"

def fetch_country(name):
    url = f"https://restcountries.com/v3.1/name/{name}?fields={FIELDS}"
    try:
        r = requests.get(url, timeout=10)
    except requests.RequestException as e:
        print("Network error:", e)
        return None
    if r.status_code != 200:
        print(f"{name}: status {r.status_code}")
        return None
    data = r.json()
    return data[0] if data else None


def print_passport(country):
    if not country:
        return
    name = country["name"]["common"]
    capital = country["capital"][0]
    pop = f"{country['population']:,}"
    langs = ", ".join(country.get("languages", {}).values())
    flag = country.get("flags", {}).get("png", "no flag")
    print(f"{name} — capital {capital}, population {pop}")
    print(f"Languages: {langs}")
    print(f"Flag: {flag}")


md = fetch_country("moldova")
print_passport(md)
print_passport(fetch_country("atlantis"))

print("\nNeighbors of Moldova:")
if md and md.get("borders"):
    for code in md["borders"]:
        n = requests.get(f"https://restcountries.com/v3.1/alpha/{code}?fields=name,population", timeout=10)
        if n.status_code == 200:
            c = n.json()
            print(f"  {c['name']['common']}: {c['population']:,}")

Path("data").mkdir(exist_ok=True)
if md:
    Path("data/moldova_passport.json").write_text(json.dumps(md, indent=2))
    print("\nsaved data/moldova_passport.json")
